In [ ]:
! pip install triton

In [ ]:
import torch
import torch.nn.functional as F
import triton
import triton.language as tl

In [ ]:
DEVICE = "cuda"
C_OUT = 64
C_IN = 3
H = 1024
W = 1024
FH = 3
FW = 3

In [ ]:
tensor_I = torch.rand(1, C_IN, H, W, device=DEVICE)          
tensor_F = torch.rand(C_OUT, C_IN, FH, FW, device=DEVICE)   

In [ ]:

golden_out = F.conv2d(tensor_I, tensor_F, padding=1)
print(golden_out.shape)  

torch.Size([1, 64, 1024, 1024])


In [ ]:

@triton.jit
def my_triton_kernel(
    input_ptr,
    filter_ptr,
    output_ptr,
    C_IN:  tl.constexpr,
    C_OUT: tl.constexpr,
    H:     tl.constexpr,
    W:     tl.constexpr,
    FH:    tl.constexpr,
    FW:    tl.constexpr,
    PAD:   tl.constexpr,
    TILE_H: tl.constexpr,
    TILE_W: tl.constexpr,
):
    
    oc    = tl.program_id(0)                    
    tile_h = tl.program_id(1)                   
    tile_w = tl.program_id(2)                   

    oh_start = tile_h * TILE_H
    ow_start = tile_w * TILE_W

    oh_offs = oh_start + tl.arange(0, TILE_H)   
    ow_offs = ow_start + tl.arange(0, TILE_W)   

    oh_2d = oh_offs[:, None]    
    ow_2d = ow_offs[None, :]    

    mask = (oh_2d < H) & (ow_2d < W)

    
    acc = tl.zeros((TILE_H, TILE_W), dtype=tl.float32)

    for ic in range(C_IN):
        for fr in range(FH):
            for fc in range(FW):
                ih = oh_2d + fr - PAD   
                iw = ow_2d + fc - PAD   

                in_mask = mask & (ih >= 0) & (ih < H) & (iw >= 0) & (iw < W)

                in_idx = ic * H * W + ih * W + iw   
                in_val = tl.load(input_ptr + in_idx, mask=in_mask, other=0.0)

                flt_idx = oc * C_IN * FH * FW + ic * FH * FW + fr * FW + fc
                flt_val = tl.load(filter_ptr + flt_idx)

                acc += in_val * flt_val

    out_idx = oc * H * W + oh_2d * W + ow_2d   
    tl.store(output_ptr + out_idx, acc, mask=mask)


def my_conv2d(input, kernel):
    
    _, c_in, h, w     = input.shape
    c_out, _, fh, fw  = kernel.shape
    pad               = fh // 2   

    output = torch.empty(1, c_out, h, w, device=input.device, dtype=input.dtype)

    TILE_H = 16
    TILE_W = 16

    grid = (
        c_out,
        triton.cdiv(h, TILE_H),
        triton.cdiv(w, TILE_W),
    )

    start_event = torch.cuda.Event(enable_timing=True)
    end_event   = torch.cuda.Event(enable_timing=True)

    my_triton_kernel[grid](
        input, kernel, output,
        c_in, c_out, h, w, fh, fw, pad,
        TILE_H, TILE_W,
    )
    torch.cuda.synchronize()

    start_event.record()
    my_triton_kernel[grid](
        input, kernel, output,
        c_in, c_out, h, w, fh, fw, pad,
        TILE_H, TILE_W,
    )
    end_event.record()
    torch.cuda.synchronize()

    execution_time_ms = start_event.elapsed_time(end_event)
    return output, execution_time_ms


In [ ]:

my_output, execution_time = my_conv2d(tensor_I, tensor_F)
torch.testing.assert_close(golden_out, my_output, atol=1e-3, rtol=1e-3)  
print(f"Execution Time for triton kernel: {execution_time:.4f} ms")

Execution Time for triton kernel: 2.8180 ms
